In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# مراقبة مهمة أو إلغاؤها

اطّلع على قائمة أحمال عملك في [صفحة Workloads](https://quantum.cloud.ibm.com/workloads).

## عرض حالة المهمة
انتقل إلى [جدول Workloads](https://quantum.cloud.ibm.com/workloads) وتحقق من عمود Status لمعرفة ما إذا كانت المهمة قد اكتملت أو فشلت.

## عرض الاستخدام المتبقي
انتقل إلى [جدول Instances](https://quantum.cloud.ibm.com/instances) واختر التبويب المرتبط بالخطة التي تريد معرفة استخدامها المتبقي. سيظهر لك إجمالي الوقت المُستخدَم والوقت المتبقي في خطتك.

## عرض مقاييس عدد المهام وأحمال العمل المُرسَلة
انتقل إلى [صفحة Analytics](https://quantum.cloud.ibm.com/analytics) لترى الإجمالي الكلي للمهام المُرسَلة، إضافةً إلى عدد أحمال عمل الدُّفعات (batch) وأحمال عمل الجلسات (session). لاحظ أنك لا تستطيع الاطلاع على صفحة Analytics إلا للحسابات التي تمتلكها أو تديرها.

## مراقبة مهمة
استخدم نسخة المهمة (job instance) للتحقق من حالتها أو استرداد نتائجها عبر استدعاء الأمر المناسب:

|                               |                                                                                                                                                                                                              |
| ----------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| job.result()                  | اعرض نتائج المهمة فور اكتمالها. تتوفر النتائج بعد انتهاء المهمة، لذا فإن `job.result()` استدعاءٌ محجوب (blocking call) حتى تكتمل المهمة.                               |
| job.job\_id()                 | أعِد المعرّف الفريد للمهمة. استرداد النتائج لاحقاً يستلزم معرفة هذا المعرّف، لذا يُنصح بحفظ معرّفات المهام التي قد تحتاج إلى استردادها لاحقاً. |
| job.status()                  | تحقق من حالة المهمة.                                                                                                                                                                                        |
| job = service.job(\<job\_id>) | استرجع مهمة سبق أن أرسلتها. يستلزم هذا الاستدعاء معرّف المهمة.                                                                                                                                      |

<span id="retrieve-later"></span>
## استرداد نتائج المهمة في وقت لاحق
استدعِ `service.job(\<job\_id>)` لاسترداد مهمة أرسلتها سابقاً. إن لم يكن لديك معرّف المهمة، أو أردت استرداد عدة مهام دفعةً واحدة — بما في ذلك المهام من وحدات المعالجة الكمية (QPUs) المُتقاعَدة — فاستخدم `service.jobs()` مع مرشّحات اختيارية. راجع [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs).

> **Note:** تُعيد `service.jobs()` أيضاً المهام التي نُفِّذت من حزمة `qiskit-ibm-provider` المُهمَلة. أما المهام التي أُرسلت عبر الحزمة الأقدم (المُهمَلة أيضاً) `qiskit-ibmq-provider` فلم تعد متاحة.

### مثال
يُعيد هذا المثال أحدث 10 مهام runtime نُفِّذت على `my_backend`:

In [ ]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

## Next steps

<Admonition type="tip" title="Recommendations">
    - Try the [Grover's algorithm](/docs/tutorials/grovers-algorithm) tutorial.
    - Learn more about [Sampler execution spans](/docs/guides/sampler-input-output#execution-spans)
</Admonition>